# Driver Drowsiness Detection using MediaPipe Face Landmarker

This notebook detects prolonged eye closure from webcam frames using current MediaPipe facial landmarks and the Eye Aspect Ratio (EAR).

## 1. Import libraries and configure thresholds

In [1]:
import time
from pathlib import Path
from urllib.request import urlretrieve

import cv2
import mediapipe as mp
import numpy as np

try:
    import winsound
except ImportError:
    winsound = None

EAR_THRESHOLD = 0.21
CLOSED_FRAME_THRESHOLD = 15
ALARM_COOLDOWN_SECONDS = 0.75

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task'
PROJECT_DIR = Path.cwd() / 'Driver_Drowsiness_Detection'
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()
MODEL_PATH = PROJECT_DIR / 'face_landmarker.task'
if not MODEL_PATH.exists():
    print('Downloading the official MediaPipe Face Landmarker model. This happens only once.')
    urlretrieve(MODEL_URL, MODEL_PATH)
    print('Model download complete.')

Model download complete.


## 2. Calculate the Eye Aspect Ratio

In [2]:
def eye_aspect_ratio(landmarks, eye_indices, frame_width, frame_height):
    points = np.array([
        (landmarks[index].x * frame_width, landmarks[index].y * frame_height)
        for index in eye_indices
    ])
    vertical_1 = np.linalg.norm(points[1] - points[5])
    vertical_2 = np.linalg.norm(points[2] - points[4])
    horizontal = np.linalg.norm(points[0] - points[3])
    if horizontal == 0:
        return 0.0, points.astype(int)
    return (vertical_1 + vertical_2) / (2.0 * horizontal), points.astype(int)


def play_alert():
    if winsound is not None:
        winsound.Beep(880, 180)
    else:
        print('\a', end='', flush=True)

## 3. Start webcam monitoring

In [3]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(MODEL_PATH)),
    running_mode=RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

closed_frames = 0
last_alarm_time = 0.0
capture = cv2.VideoCapture(0)
if not capture.isOpened():
    raise RuntimeError('Could not open the webcam. Check camera permissions and try again.')

try:
    with FaceLandmarker.create_from_options(options) as landmarker:
        while capture.isOpened():
            success, frame = capture.read()
            if not success:
                print('Could not read a frame from the webcam.')
                break

            frame = cv2.flip(frame, 1)
            height, width = frame.shape[:2]
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            timestamp_ms = int(time.monotonic() * 1000)
            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            status, status_color = 'Face not detected', (0, 165, 255)
            if result.face_landmarks:
                landmarks = result.face_landmarks[0]
                left_ear, left_points = eye_aspect_ratio(landmarks, LEFT_EYE, width, height)
                right_ear, right_points = eye_aspect_ratio(landmarks, RIGHT_EYE, width, height)
                average_ear = (left_ear + right_ear) / 2.0

                cv2.polylines(frame, [left_points], True, (0, 255, 0), 1)
                cv2.polylines(frame, [right_points], True, (0, 255, 0), 1)

                if average_ear < EAR_THRESHOLD:
                    closed_frames += 1
                    status, status_color = 'Eyes closed', (0, 0, 255)
                else:
                    closed_frames = 0
                    status, status_color = 'Eyes open', (0, 255, 0)

                if closed_frames >= CLOSED_FRAME_THRESHOLD:
                    status, status_color = 'DROWSINESS ALERT!', (0, 0, 255)
                    if time.monotonic() - last_alarm_time >= ALARM_COOLDOWN_SECONDS:
                        play_alert()
                        last_alarm_time = time.monotonic()

                cv2.putText(frame, f'EAR: {average_ear:.2f}', (20, 35),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

            cv2.putText(frame, status, (20, 70), cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, status_color, 2)
            cv2.putText(frame, f'Closed frames: {closed_frames}/{CLOSED_FRAME_THRESHOLD}',
                        (20, 105), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.imshow('Driver Drowsiness Detection', frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
finally:
    capture.release()
    cv2.destroyAllWindows()

## Expected behaviour

- Eyes open: the counter stays at zero.
- A normal blink: the counter resets before the alert threshold.
- Eyes closed for the configured number of frames: the alert displays and an audio tone plays.
- Press q to close the camera window.